# домашняя работа №2

*курс по обработке естественного языка, ВШЭ 2026*

### General Rules and Submission Guidelines

1. Copying code from external sources (**including using LLMs**) without explicit citation is strictly prohibited and will result in 0 points for the entire assignment. If you consult any resources or AI tools, you must clearly state this in a separate Markdown cell. If suspected of this, you might be asked to explain your code to the grader and answer their questions during a separate session to avoid the mentioned penalty.
2. All results must be fully **reproducible**. You are required to use `set_seed` everywhere so that the grader can obtain the same results when rerunning your notebook.
3. The notebook must run from top to bottom without errors. Submissions that fail to execute sequentially will not be accepted.
4. You must satisfy all requirements in each task to receive full credit. Partial completion may lead to partial scoring.
5. Do not modify the original notebook structure or provided Markdown cells. You are only allowed to write code in the sections marked `# TODO: your code here`. Any explanations, interpretations, or additional comments must be placed **in separate Markdown cells**. If you choose to do so, leave an explanation as to what and why was changed.
6. The final submission must be a completed `.ipynb` Jupyter Notebook. You may conduct your experiments in Jupyter Notebook, VS Code, or Google Colab — whichever environment you prefer.

**Note: You may write your textual comments in Markdown for the conducted experiments *in either Russian or English***.

In [ ]:
# подключение Google Drive и папка для результатов (для Colab)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_DIR = "/content/drive/MyDrive/NLP_HW2_results"
except Exception:
    RESULTS_DIR = "./NLP_HW2_results"
import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Results will be saved to:", RESULTS_DIR)

### настройка окружения

подключаем нужные библиотеки. если чего-то не хватает - можно добавить потом.

In [ ]:
# финальный прогон: весь dev (983 примера) для шагов 3–6
DEBUG = False
SEED = 42
DEBUG_SAMPLE_SIZE = 5
FINAL_SAMPLE_SIZE = 983

import random
import numpy as np
import torch
from transformers import set_seed as hf_set_seed

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
hf_set_seed(SEED)
print("DEBUG =", DEBUG, "| SEED =", SEED)

In [ ]:
# утилиты сохранения результатов (RESULTS_DIR задаётся в ячейке с Drive)
import json
from datetime import datetime

if "RESULTS_DIR" not in dir():
    RESULTS_DIR = "./NLP_HW2_results"
    import os
    os.makedirs(RESULTS_DIR, exist_ok=True)

def save_df(df, name):
    path = os.path.join(RESULTS_DIR, f"{name}.csv")
    df.to_csv(path, index=False)
    print("Saved:", path)

def save_json(obj, name):
    path = os.path.join(RESULTS_DIR, f"{name}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    print("Saved:", path)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)

from datasets import Dataset
import evaluate
import optuna
import random

# что делаем: выбираем, где считать - видеокарта или процессор; фиксируем сиды, чтобы результаты повторялись
device = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)
print("device:", device)

device: cpu


### **0. Introduction to the Task and Data**

In this assignment, you will get acquainted with a binary text classification task in Russian, known as Linguistic Acceptability. Your goal will be to train a model capable of distinguishing grammatically correct sentences from incorrect ones.

**RuCoLA (Russian Corpus of Linguistic Acceptability)** is a corpus of Russian sentences annotated with a binary label indicating whether a sentence is linguistically acceptable (grammatically correct and meaningful) or not.

**Structure:** The data is already split into training, validation, and test sets. A detailed description and the dataset itself are available in the project's official [repository](https://github.com/RussianNLP/RuCoLA).

***Important:*** Before starting the assignment, carefully study the repository's `README`. Pay special attention to the sections describing the data structure and the already implemented baselines — this will help you in your work.

### **1. Data Preparation (0.25 point)**



**Data Location:**
The data is located in the [repository](https://github.com/RussianNLP/RuCoLA/tree/main/data). Download the files contained in this folder.




**Files to be Used:**
*   For **training** (`train`), we will use the file **`in_domain_train.csv`**. This file contains examples from linguistic sources.
*   For **testing** (`test`), we will use the file **`in_domain_dev.csv`**.






**Data Format:**
The essential information for us is located in two columns:
*   **`sentence`**: This column contains the text of the sentence (input data for the model).
*   **`acceptable`**: This is the target variable for prediction. It contains a binary label:
    *   **1** — the sentence is linguistically acceptable (grammatically correct).
    *   **0** — the sentence is unacceptable (contains an error).

By the end of this step, you should **have both training and test datasets loaded and ready for further preprocessing** and model experimentation.

In [2]:
# TODO: your code here
#загружаем обучающую и проверочную выборки из csv  
train_df = pd.read_csv("in_domain_train.csv")
test_df = pd.read_csv("in_domain_dev.csv")

#оставляем только текст и целевую метку
train_df = train_df[["sentence", "acceptable"]]
test_df = test_df[["sentence", "acceptable"]]

# готовим отдельные объекты с текстами и метками
# дальше будем подавать тексты в токенайзер, а метки - в модель
X_train = train_df["sentence"].tolist()
y_train = train_df["acceptable"].tolist()
X_test = test_df["sentence"].tolist()
y_test = test_df["acceptable"].tolist()

# проверим размеры выборок
print("размер train:", train_df.shape)
print("размер test:", test_df.shape)

# проверим, есть ли пропуски в текстах
print("пропуски в train по sentence:", train_df["sentence"].isna().sum())
print("пропуски в test по sentence:", test_df["sentence"].isna().sum())

размер train: (7869, 2)
размер test: (983, 2)
пропуски в train по sentence: 0
пропуски в test по sentence: 0


In [ ]:
# общая подвыборка для шагов 3–6: одна и та же для честного сравнения
if DEBUG:
    X_run = list(X_test[:DEBUG_SAMPLE_SIZE])
    y_run = list(y_test[:DEBUG_SAMPLE_SIZE])
else:
    X_run = list(X_test[:FINAL_SAMPLE_SIZE])
    y_run = list(y_test[:FINAL_SAMPLE_SIZE])
print("X_run size:", len(X_run), "| y_run size:", len(y_run))

# логирование конфигурации эксперимента
experiment_config = f"""
seed = {SEED}
debug_mode = {DEBUG}
debug_sample_size = {DEBUG_SAMPLE_SIZE}
final_sample_size = {FINAL_SAMPLE_SIZE}
device = {device}
run_size = {len(X_run)}
"""
with open(os.path.join(RESULTS_DIR, "experiment_config.txt"), "w", encoding="utf-8") as f:
    f.write(experiment_config)
print("Saved:", os.path.join(RESULTS_DIR, "experiment_config.txt"))

Шаг 2 оценивается на полном dev (in_domain_dev.csv). Шаги 3–6 используют X_run / y_run; при финальном конфиге (DEBUG=False, FINAL_SAMPLE_SIZE=983) X_run совпадает с полным dev (983 примера), т.е. все шаги в финальном прогоне считаются на одном объёме.

In [3]:
# посмотрим распределение классов
print("распределение классов в train:")
print(train_df["acceptable"].value_counts())
print("доли классов в train:")
print(train_df["acceptable"].value_counts(normalize=True))

print("распределение классов в test:")
print(test_df["acceptable"].value_counts())
print("доли классов в test:")
print(test_df["acceptable"].value_counts(normalize=True))

распределение классов в train:
acceptable
1    5864
0    2005
Name: count, dtype: int64
доли классов в train:
acceptable
1    0.745203
0    0.254797
Name: proportion, dtype: float64
распределение классов в test:
acceptable
1    733
0    250
Name: count, dtype: int64
доли классов в test:
acceptable
1    0.745677
0    0.254323
Name: proportion, dtype: float64


In [4]:
# посмотрим на первые строки train_df
print("первые строки train_df:")
print(train_df.head())

первые строки train_df:
                                            sentence  acceptable
0  Вдруг решетка беззвучно поехала в сторону, и н...           1
1                       Этим летом не никуда ездили.           0
2  Только Иван выразил какую бы то ни было готовн...           1
3  Теперь ты видишь собственными глазами, как тут...           1
4    На поверку вся теория оказалась полной чепухой.           1


**Evaluation Metrics:**
The performance of your models will be evaluated using:

1.  **Accuracy:** The percentage of sentences for which the model correctly predicted the label.
2.  **Matthews Correlation Coefficient (MCC):** This is a more robust metric for binary classification tasks, especially if the classes might be slightly imbalanced. It takes into account all four categories of the confusion matrix (True Positives, True Negatives, False Positives, False Negatives) and returns a value in the range of -1 to +1.

**Important:** You *do not need to implement MCC* from scratch. Import and use the ready-made function from the scikit-learn library:

In [5]:
from sklearn.metrics import matthews_corrcoef

### **2. Baseline: Fine-tuning ruBERT (1 point)**


In this section, you will implement a classifier based on one of the BERT family models adapted for the Russian language. Perform fine-tuning of the model on the training data `in_domain_train.csv` and evaluate its quality on `in_domain_dev.csv`.

**What you need to do:**

1.  **Model Selection (0.25 point)**

For the experiment, you can choose one of the following pre-trained models from the Hugging Face Hub (they differ in size and speed):
*   [`cointegrated/rubert-tiny2`](https://huggingface.co/cointegrated/rubert-tiny2)
*   [`ai-forever/ruBert-base`](https://huggingface.co/ai-forever/ruBert-base)
*   [`ai-forever/ruBert-large`](https://huggingface.co/ai-forever/ruBert-large)



In [ ]:
# загружаем токенайзер для выбранной модели чтобы перевести тексты в числа, понятные модели
model_name = "cointegrated/rubert-tiny2"
print(model_name)

tokenizer = AutoTokenizer.from_pretrained(model_name)

# подготовить входы для модели
train_encodings = tokenizer(
    X_train,
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt",
)

test_encodings = tokenizer(
    X_test,
    truncation=True,
    padding=True,
    max_length=128,
    return_tensors="pt",
)

# чтобы Trainer мог удобно читать данные из наших списков
class RuCoLADataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item


train_dataset = RuCoLADataset(train_encodings, y_train)
test_dataset = RuCoLADataset(test_encodings, y_test)


мы будем использовать модель: cointegrated/rubert-tiny2
датасеты для transformers готовы


2.  **Data Preparation (0.25 point)**
    
    Convert the texts and labels into a format suitable for `transformers`. Use the tokenizer corresponding to your chosen model.

In [8]:
#  загружаем модель и настраиваем обучение с ранней остановкой
# дообучить rubert под задачу и не переобучиться
# если в памяти остались старые датасеты (не из datasets) - пересоздаём, чтобы Trainer не падал
from datasets import Dataset as HFDataset
from sklearn.metrics import accuracy_score, matthews_corrcoef

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "matthews_correlation": matthews_corrcoef(labels, preds),
    }

if not hasattr(train_dataset, "column_names"):
    def _tok(batch):
        return tokenizer(batch["sentence"], truncation=True, padding=True, max_length=128)
    train_dataset = HFDataset.from_dict({"sentence": X_train, "labels": y_train}).map(_tok, batched=True)
    test_dataset = HFDataset.from_dict({"sentence": X_test, "labels": y_test}).map(_tok, batched=True)

set_seed(42)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
)
model.to(device)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="rubert_rucola_baseline",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_strategy="epoch",
)

early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
    callbacks=[early_stopping],
    compute_metrics=compute_metrics,
)

train_result = trainer.train()

print("обучение остановилось на эпохе:", trainer.state.epoch)

# сохраняем лучшую версию модели
trainer.save_model("rubert_rucola_best")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Map:   0%|          | 0/7869 [00:00<?, ? examples/s]

Map:   0%|          | 0/983 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/p4/qk6t1wt14b9dhc4806cz9n880000gn/T/ipykernel_49606/2766251973.py:38: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/Users/sofya/kachectvo-zadachi-7054/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,0.543400,0.534291
2,0.492000,0.513105
3,0.418200,0.551780
4,0.342600,0.612352


/Users/sofya/kachectvo-zadachi-7054/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/sofya/kachectvo-zadachi-7054/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/sofya/kachectvo-zadachi-7054/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


обучение остановилось на эпохе: 4.0


3. **Fine-tuning (0.25 point)**

Stopping Criterion: Train the model until it reaches a plateau (early stopping by loss). Once the loss stops improving for several epochs (e.g., 2-3), training should be stopped. This helps prevent overfitting.
Record the number of epochs it took to reach the plateau.

In [ ]:
# Вычисление метрик — в разделе 2.4 Quality Evaluation

/Users/sofya/kachectvo-zadachi-7054/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Accuracy: 0.780264496439471
MCC: 0.31679782101807685
обучение фактически остановилось на эпохе: 4.0


4. **Quality Evaluation (0.25 point)**

After training is complete, make predictions for the entire in_domain_dev.csv file.
Calculate the Accuracy and MCC metrics (using sklearn.metrics.matthews_corrcoef).
Compare the obtained results and comment on them in a Markdown cell.

In [ ]:
# TODO: your code here
from sklearn.metrics import accuracy_score, matthews_corrcoef

predictions = trainer.predict(test_dataset)
logits = predictions.predictions
y_true = predictions.label_ids
y_pred = logits.argmax(axis=-1)

accuracy = accuracy_score(y_true, y_pred)
mcc = matthews_corrcoef(y_true, y_pred)

print("Accuracy:", accuracy)
print("MCC:", mcc)

df_rubert_metrics = pd.DataFrame([{"accuracy": accuracy, "MCC": mcc}])
save_df(df_rubert_metrics, "step2_rubert_metrics")

**Комментарий к результатам.**  
Accuracy на in_domain_dev составила [accuracy], MCC — [MCC]. Обучение остановилось на эпохе [N]. Краткий вывод: [насколько хорошо модель отделяет приемлемые предложения от неприемлемых; при необходимости — сравнение с бейзлайном или ожиданиями].

### **3. Zero-shot Approach Using a Generative Model (ruGPT-3) (4.25 point)**


In this section, we will try to solve the task without any training, using only a pre-trained generative language model — **ruGPT-3 Large**.



**Model:** Use [`ai-forever/rugpt3large_based_on_gpt2`](https://huggingface.co/ai-forever/rugpt3large_based_on_gpt2) from the `transformers` library.



**Task:**

1.  **Implement a function to calculate loss for a single text (1.25 point)**

You will need to write a function that:
*   Tokenizes the input text, preparing it for the model (don't forget `return_tensors="pt"` and moving it to the `device`).
*   Passes the tokenized inputs to the model, specifying `labels` (for GPT-2/3 models, labels are usually the same as `input_ids`, since the task is next-token prediction).
*   Extracts the `loss` value from the model's output.
*   Returns the loss value (a number).

    **Important:** To complete this task, you can use [this Colab notebook](https://colab.research.google.com/drive/1BnfhKQQiNAXrKlkIynVPFleQ-epF3I55#scrollTo=fOemj3PbeCZi) and the `transformers` documentation as a reference.



In [ ]:
tokenizer = AutoTokenizer.from_pretrained("ai-forever/rugpt3large_based_on_gpt2")
model = AutoModelForCausalLM.from_pretrained("ai-forever/rugpt3large_based_on_gpt2")
# при нехватке памяти можно заменить на ai-forever/rugpt3small_based_on_gpt2
model.to(device)


In [ ]:
def get_loss_num(text):
    # что делаем: считаем loss для одного текста
    # зачем: потом сравним loss для «правильное» и «неправильное» и решим класс
    inputs = tokenizer(text, return_tensors="pt")
    input_ids = inputs["input_ids"].to(device)
    with torch.no_grad():
        outputs = model(input_ids=input_ids, labels=input_ids)
    return outputs.loss.item()

2.  **Implement the prediction function `predict_zero_shot(text, pos_template, neg_template)` (1 point)**
    *   The function takes the original text and two templates (prompts) — one for the positive (acceptable) class and one for the negative (unacceptable) class.
    *   The templates should be strings with a placeholder `{}`, where the original text will be inserted. For example: `"This sentence is grammatically correct: {}"`.
    *   Inside the function, calculate the loss for the text inserted into the positive template (`pos_loss`) and the loss for the text inserted into the negative template (`neg_loss`).
    *   Compare the values: if `pos_loss` is **less than** `neg_loss` (the model finds the positive variant more probable), return `1` (the "acceptable" class); otherwise, return `0`.

In [ ]:
def predict_zero_shot(sentence, pos_template, neg_template):
    # что делаем: подставляем предложение в два шаблона и сравниваем loss
    # зачем: меньший loss = модель считает вариант более вероятным
    pos_text = pos_template.format(sentence)
    neg_text = neg_template.format(sentence)
    pos_loss = get_loss_num(pos_text)
    neg_loss = get_loss_num(neg_text)
    if pos_loss < neg_loss:
        return 1
    return 0

3.  **Experiments with prompts and generation parameters (1 point)**

This part consists of two sub-tasks: first, you will find the best prompt pair using a fixed decoding strategy. Then, you will experiment with different generation parameters using that best prompt.

Fix the decoding strategy to greedy search. Use this setting for all prompt evaluations.
Create at least three different pairs of prompts. Vary the wording, punctuation, level of detail, and whether the prompt is a statement, question, or instruction.

For each prompt compute Accuracy and MCC.

Identify the prompt pair that yields the highest performance. This will be your best prompt for the next sub-task.

In [ ]:
# минимум 3 пары шаблонов: короткие, подлиннее, с инструкцией
# для каждой пары считаем предсказания на X_test и метрики
from sklearn.metrics import accuracy_score, matthews_corrcoef

set_seed(42)

prompt_pairs = [
    ("короткая", "Грамматически правильное предложение: {}", "Грамматически неправильное предложение: {}"),
    ("подробная", "Определи, является ли предложение грамматически правильным. Правильное предложение: {}", "Определи, является ли предложение грамматически правильным. Неправильное предложение: {}"),
    ("утверждение", "Предложение корректно: {}", "Предложение некорректно: {}"),
]

results_prompts = []
for name, pos_t, neg_t in prompt_pairs:
    y_pred = [predict_zero_shot(s, pos_t, neg_t) for s in X_run]
    acc = accuracy_score(y_run, y_pred)
    mcc = matthews_corrcoef(y_run, y_pred)
    results_prompts.append({"prompt_pair": name, "accuracy": acc, "MCC": mcc})

df_prompts = pd.DataFrame(results_prompts)
print("Результаты по парам промптов:")
print(df_prompts.to_string(index=False))
save_df(df_prompts, "step3_prompt_results")

best_row = df_prompts.loc[df_prompts["MCC"].idxmax()]
best_pair_name = best_row["prompt_pair"]
best_pos = next(p for n, p, neg in prompt_pairs if n == best_pair_name)
best_neg = next(neg for n, p, neg in prompt_pairs if n == best_pair_name)
print("\nЛучшая пара промптов:", best_pair_name)

Using the best prompt from previous step, experiment with at least three different generation strategies. The loss calculation can be affected by how the model processes the input. You can modify the generation parameters even though you are only computing loss

For each configuration, evaluate on the test set and record Accuracy and MCC

In [ ]:
# 3.4 эксперименты с decoding strategy: используем лучшую prompt-пару и генерацию с разными настройками
import re

set_seed(42)

def extract_01(text):
    m = re.search(r"[01]", text)
    return int(m.group()) if m else 0

def predict_by_generation(sentence, pos_template, neg_template, gen_kwargs):
    prompt = pos_template.format(sentence) + "\nМетка:"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=5,
            pad_token_id=tokenizer.eos_token_id,
            **gen_kwargs,
        )
    new = out[0][inputs["input_ids"].shape[1]:]
    return extract_01(tokenizer.decode(new, skip_special_tokens=True))

strategies = [
    ("greedy", {"do_sample": False}),
    ("beam_search", {"do_sample": False, "num_beams": 3}),
    ("sampling_top_p", {"do_sample": True, "top_p": 0.9}),
]
results_decode = []
for strat_name, kw in strategies:
    set_seed(42)
    y_pred = [predict_by_generation(s, best_pos, best_neg, kw) for s in X_run]
    acc = accuracy_score(y_run, y_pred)
    mcc = matthews_corrcoef(y_run, y_pred)
    results_decode.append({"decoding_strategy": strat_name, "accuracy": acc, "MCC": mcc})

df_decode = pd.DataFrame(results_decode)
print("Результаты по стратегиям декодирования:")
print(df_decode.to_string(index=False))
save_df(df_decode, "step3_decoding_results")


4.  **Analysis of Results (1 point)**
    *   Compare the metrics obtained with different prompts.
    *   Draw conclusions about how the following factors influence quality:
        *   The wording of the task (statement, question, instruction).
        *   The presence of punctuation (a period at the end, a question mark).
        *   The length and specificity of the prompt.
    *   Try to explain why one pair of prompts worked better than another.

**3.5 Анализ результатов**

- **Лучшая пара промптов:** по результатам выше лучший результат показала пара [best prompt pair]: MCC составил [MCC], accuracy — [accuracy].
- **Влияние формулировки:** [утверждение / вопрос / инструкция] дал [лучший / сопоставимый] результат; [кратко, почему, по вашим данным].
- **Пунктуация и длина:** [короткий / подробный] промпт [улучшил / не улучшил] качество по сравнению с остальными; влияние пунктуации [заметно / слабое].
- **Стратегия декодирования:** по таблице по стратегиям лучшей оказалась [greedy / beam_search / sampling_top_p]; [одно предложение — почему].

### **4. Instruction-Tuned Model: Qwen2.5-Instruct (2 points)**



In this section, you will work with the **Qwen2.5-Instruct** model. Your goal is to select prompts that will allow the model to solve the binary classification task of linguistic acceptability without any fine-tuning, relying solely on its ability to follow instructions.



**Model:**
*   Use the base instruction-tuned version: [`Qwen/Qwen2.5-1.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct).
*   *Note:* You may also experiment with larger versions (e.g., `Qwen/Qwen2.5-7B-Instruct`) and compare the results.


**Task:**

1.  **Design a prompt for the task (0.5 point)**

Create at least three different prompts that set various contexts, roles, or behavioral constraints for the model and ask the model to classify a sentence as acceptable (1) or unacceptable (0). The prompt should be passed to the model together with the input sentence.

**!Do not use a system prompt!** in this part - only a single user message or a direct instruction.


In [ ]:
# 4.1 загрузка Qwen2.5-Instruct и функция генерации ответа (только user prompt, без system)
qwen_name = "Qwen/Qwen2.5-1.5B-Instruct"
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_name)
qwen_model = AutoModelForCausalLM.from_pretrained(qwen_name, trust_remote_code=True)
qwen_model.to(device)


def generate_answer(prompt):
    # один user message, без system prompt
    messages = [{"role": "user", "content": prompt}]
    text = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = qwen_model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    # декодируем только сгенерированную часть
    new_tokens = out[0][inputs["input_ids"].shape[1] :]
    return qwen_tokenizer.decode(new_tokens, skip_special_tokens=True)


# 4.2 три варианта user prompt
PROMPT_1_RU = """Классифицируй предложение.
Ответь 1 если оно грамматически правильное.
Ответь 0 если неправильное.
Ответь только числом.

Предложение: {}"""

PROMPT_2_EN = """Classify the sentence.
Return 1 if it is grammatically correct.
Return 0 if it is incorrect.
Return only 0 or 1.

Sentence: {}"""

PROMPT_3_FEWSHOT = """Определи, является ли предложение грамматически корректным.

Пример:
Предложение: Я люблю читать книги.
Ответ: 1

Теперь оцени:

Предложение: {}
Ответ:"""

2.  **Implement response post-processing (0.5 point)**

The model generates free text, so you need to extract the predicted label (0 or 1) from its output. Write a function that:
    *   Takes the generated model text as input.
    *   Extracts the integer 0 or 1 from it. Handle cases where the model provides additional explanations (e.g., look for the first occurrence of "1" or "0", use simple regular expressions).
    *   If the response is ambiguous, decide on a processing strategy. Describe your approach.


In [ ]:
# 4.3 извлечение метки 0 или 1 из текста ответа модели
import re


def extract_label(text):
    # изолированные 0 или 1 (граница слова), чтобы не спутать с "10" и т.п.
    m = re.search(r"\b([01])\b", text)
    if m is not None:
        return int(m.group(1))
    return 0


# проверка на типичных форматах ответа модели
assert extract_label("Ответ: 1") == 1
assert extract_label("1\n") == 1
assert extract_label("10") == 0  # нет изолированной 0/1 — возвращаем 0 по умолчанию
assert extract_label("Метка: 0") == 0
assert extract_label("0") == 0

3.  **Experiments with prompts (0.5 point)**

Conduct experiments by varying prompt characteristics. For each variant, evaluate the model on the entire `in_domain_dev.csv` file and compute **Accuracy** and **MCC**. Consider at least the following dimensions:



  *   **Prompt Language:** prompts in Russian vs. prompts in English. How does language proficiency affect the result?
  *   **Instruction Style:** compare different formulations.
  *   **Prompt Length and Detail:** short vs. detailed instructions; you can try adding one example (few-shot).




In [ ]:
# 4.4–4.5 прогоны по X_run для каждого prompt, метрики, таблица (X_run задан в ячейке конфига)
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, matthews_corrcoef

set_seed(42)

prompts_config = [
    ("русский zero-shot", PROMPT_1_RU),
    ("английский zero-shot", PROMPT_2_EN),
    ("русский few-shot", PROMPT_3_FEWSHOT),
]

results_qwen = []
raw_outputs_step4 = []
for prompt_type, template in prompts_config:
    preds = []
    for idx, sentence in enumerate(tqdm(X_run, desc=prompt_type)):
        prompt = template.format(sentence)
        answer = generate_answer(prompt)
        label = extract_label(answer)
        preds.append(label)
        raw_outputs_step4.append({
            "sentence": sentence,
            "gold": int(y_run[idx]),
            "model_answer": answer,
            "parsed_label": label,
            "prompt_type": prompt_type,
        })
    acc = accuracy_score(y_run, preds)
    mcc = matthews_corrcoef(y_run, preds)
    results_qwen.append({"prompt_type": prompt_type, "accuracy": acc, "MCC": mcc})

df_qwen = pd.DataFrame(results_qwen)
print("Результаты Qwen2.5-Instruct ({} предложений):".format(len(X_run)))
print(df_qwen.to_string(index=False))
save_df(df_qwen, "step4_qwen_results")
save_json(raw_outputs_step4, "step4_raw_outputs")


4.  **Analysis of Results (0.5 point)**
    *   Compare the quality of different prompts. Which ones work better and why?
    *   Draw conclusions about the influence of the instruction language.
    *   Provide your thoughts on whether the model size is sufficient to solve this task with good quality.
    *   Describe your findings in MarkDown.

**Анализ результатов (4.6)**

- **Лучший user prompt:** лучший результат по MCC дал вариант [best prompt]: MCC = [MCC], accuracy = [accuracy].
- **Влияние языка инструкции:** [русский / английский] промпт [оказался лучше / дал сопоставимые результаты]; [одно предложение — почему].
- **Few-shot vs zero-shot:** few-shot [улучшил / не улучшил] качество по сравнению с zero-shot; [краткое объяснение по вашим числам].
- **Достаточность размера модели (1.5B):** по метрикам модель [достаточно / недостаточно] справляется с задачей без дообучения; для сравнения, ruBERT из шага 2 на полном dev дал [указать при необходимости].

### **5. Adding a System Prompt (0,75 point)**


Add a System Prompt and **conduct an experiment similar to** **part 4**, varying the System Prompt. Draw conclusions about the influence of the System Prompt. Create **at least three different system prompts** that set various contexts, roles, or behavioral constraints for the model.

In [ ]:
# 5.1 лучший user prompt из шага 4 (по MCC)
best_row = df_qwen.loc[df_qwen["MCC"].idxmax()]
best_user_prompt = next(t for name, t in prompts_config if name == best_row["prompt_type"])
print("Лучший user prompt из шага 4:", best_row["prompt_type"])

# 5.2 три system prompt
SYSTEM_PROMPT_1 = "Ты строгий классификатор. Отвечай только 0 или 1."
SYSTEM_PROMPT_2 = "Ты лингвист. Никаких объяснений. Только 0/1."
SYSTEM_PROMPT_3 = "Ты проверяешь грамматику. Ответ в формате: LABEL: 0/1."

# 5.3 генерация ответа с system prompt (переиспользуем модель и токенайзер из шага 4)
def generate_answer_with_system(system_prompt, user_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    text = qwen_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = qwen_model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=qwen_tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1] :]
    return qwen_tokenizer.decode(new_tokens, skip_special_tokens=True)

# 5.4 extract_label уже есть в шаге 4 — используем её

# 5.5–5.6 прогон по данным и метрики для каждого system prompt (X_run из конфига)
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, matthews_corrcoef

set_seed(42)

system_prompts = [
    ("строгий классификатор 0/1", SYSTEM_PROMPT_1),
    ("лингвист только 0/1", SYSTEM_PROMPT_2),
    ("LABEL: 0/1", SYSTEM_PROMPT_3),
]

results_system = []
raw_outputs_step5 = []
for sp_type, system_prompt in system_prompts:
    preds = []
    for idx, sentence in enumerate(tqdm(X_run, desc=sp_type)):
        user_prompt = best_user_prompt.format(sentence)
        answer = generate_answer_with_system(system_prompt, user_prompt)
        label = extract_label(answer)
        preds.append(label)
        raw_outputs_step5.append({
            "sentence": sentence,
            "gold": int(y_run[idx]),
            "model_answer": answer,
            "parsed_label": label,
            "prompt_type": sp_type,
        })
    acc = accuracy_score(y_run, preds)
    mcc = matthews_corrcoef(y_run, preds)
    results_system.append({"system_prompt_type": sp_type, "accuracy": acc, "MCC": mcc})

df_system = pd.DataFrame(results_system)
print("\nРезультаты с разными system prompt ({} предложений):".format(len(X_run)))
print(df_system.to_string(index=False))
save_df(df_system, "step5_qwen_system_results")
save_json(raw_outputs_step5, "step5_raw_outputs")

best_system_row = df_system.loc[df_system["MCC"].idxmax()]
best_system_prompt = next(sp for name, sp in system_prompts if name == best_system_row["system_prompt_type"])
best_configs = {
    "step3_best_prompt_pair": best_pair_name,
    "step4_best_user_prompt": best_user_prompt,
    "step5_best_system_prompt": best_system_prompt,
}
save_json(best_configs, "best_prompts")

**5.7 Анализ результатов**

- **Лучший system prompt:** среди трёх вариантов лучший результат дал [name]: MCC = [MCC], accuracy = [accuracy].
- **Сравнение с шагом 4:** добавление system prompt [улучшило / не улучшило] метрики по сравнению с лучшим user prompt из шага 4; разница по MCC [значительная / небольшая].
- **Удобство формата для модели:** вариант с [явным форматом «только 0/1» / ролью «классификатор» / форматом «LABEL: 0/1»] оказался [наиболее эффективным / сопоставимым]; [одно предложение — почему].
- **Влияние формулировки system prompt:** разброс метрик между тремя вариантами [велик / невелик], то есть формулировка system prompt [существенно влияет / влияет слабо] на результат.

### **6. Base (Non-Instruction) Version of Qwen2.5 (0,75 point)**


Conduct an experiment similar to the approach in section 4-5, but using the base non-instruction version of the **Qwen/Qwen2.5-1.5B** model. Use [`Qwen/Qwen2.5-1.5B`](https://huggingface.co/Qwen/Qwen2.5-1.5B) (or `Qwen/Qwen2.5-7B`).

Use exactly the same hyperparameters, generation strategy, and system prompts that turned out to be optimal for the instruction-tuned version.

In [ ]:
# 6.1 лучший сетап из шага 5: user prompt, system prompt, те же generation params
best_system_row = df_system.loc[df_system["MCC"].idxmax()]
best_system_prompt = next(sp for name, sp in system_prompts if name == best_system_row["system_prompt_type"])
print("Лучший system prompt из шага 5:", best_system_row["system_prompt_type"])
# best_user_prompt уже есть из шага 5; params: max_new_tokens=5, do_sample=False, temperature=0

# 6.2 загрузка base-модели (отдельные переменные, instruct не трогаем)
qwen_base_name = "Qwen/Qwen2.5-1.5B"
qwen_base_tokenizer = AutoTokenizer.from_pretrained(qwen_base_name)
qwen_base_model = AutoModelForCausalLM.from_pretrained(qwen_base_name, trust_remote_code=True)
qwen_base_model.to(device)

# 6.3 генерация с base-моделью — та же логика, что в шаге 5
def generate_answer_with_system_base(system_prompt, user_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    text = qwen_base_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = qwen_base_tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = qwen_base_model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=qwen_base_tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1] :]
    return qwen_base_tokenizer.decode(new_tokens, skip_special_tokens=True)

# 6.4 прогон по тем же данным (X_run из конфига)
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, matthews_corrcoef

set_seed(42)

preds_base = []
raw_outputs_step6 = []
for idx, sentence in enumerate(tqdm(X_run, desc="Qwen2.5-1.5B-Base")):
    user_prompt = best_user_prompt.format(sentence)
    answer = generate_answer_with_system_base(best_system_prompt, user_prompt)
    label = extract_label(answer)
    preds_base.append(label)
    raw_outputs_step6.append({
        "sentence": sentence,
        "gold": int(y_run[idx]),
        "model_answer": answer,
        "parsed_label": label,
        "model_type": "Qwen2.5-1.5B-Base",
    })

base_acc = accuracy_score(y_run, preds_base)
base_mcc = matthews_corrcoef(y_run, preds_base)

# 6.5 сравнительная таблица: instruct (лучший из шага 5) vs base на одном подмножестве
instruct_acc = best_system_row["accuracy"]
instruct_mcc = best_system_row["MCC"]
df_compare = pd.DataFrame([
    {"model_type": "Qwen2.5-1.5B-Instruct", "accuracy": instruct_acc, "MCC": instruct_mcc},
    {"model_type": "Qwen2.5-1.5B-Base", "accuracy": base_acc, "MCC": base_mcc},
])
print("Сравнение на одних и тех же {} предложениях:".format(len(X_run)))
print(df_compare.to_string(index=False))
save_df(df_compare, "step6_qwen_base_vs_instruct")
save_json(raw_outputs_step6, "step6_raw_outputs")

**6.6 Анализ результатов**

1. **Сравнение instruct и base:** на одной и той же подвыборке instruct-модель дала accuracy [value], MCC [value]; base-модель — accuracy [value], MCC [value]. [Instruct / Base] показала лучший результат.
2. **Насколько base хуже:** разница по MCC составила [value]; по accuracy — [value]. Base [заметно уступает / держится близко к instruct].
3. **Соблюдение формата «только 0/1»:** base-модель [часто / редко] выдавала лишний текст; после извлечения метки через extract_label качество [приемлемое / заметно страдает].
4. **Лишний текст у base:** в raw_outputs видно, что ответы base [часто длинные и с объяснениями / в основном краткие]; это [влияет / слабо влияет] на итоговые метрики.
5. **Почему instruct предпочтительнее:** instruction fine-tuning учит модель следовать формату ответа и инструкциям, поэтому instruct стабильнее возвращает 0/1; base к этому не приучена и чаще генерирует произвольный текст, что хуже подходит для acceptability classification.

### **7. Draw conclusions (1 point)**


Describe whether the results of the instruction-tuned and non-instruction versions of the same base model differ. Explain what these differences (or lack thereof) are related to. How does instruction fine-tuning affect the model's ability to solve the linguistic acceptability task?

**Выводы.**

- **Различие instruct и base:** на одной подвыборке instruct-модель (Qwen2.5-1.5B-Instruct) показала accuracy [value], MCC [value]; base (Qwen2.5-1.5B) — accuracy [value], MCC [value]. Результаты [различаются заметно / близки]; различие связано с тем, что [кратко].
- **Влияние instruction fine-tuning:** дообучение на инструкциях улучшает способность модели решать задачу acceptability classification: модель чаще отвечает в формате 0/1, меньше генерирует лишний текст и лучше следует system/user prompt. Base-модель без такого дообучения [часто даёт длинные ответы / хуже держит формат], что [снижает метрики / усложняет парсинг].
- **Формат ответа:** для задачи бинарной классификации пригодность instruct выше: явный запрос «ответь только 0 или 1» и system prompt с ограничением формата работают стабильнее на instruct; base склонна к развёрнутым ответам, из которых метку приходится извлекать эвристиками.
- **Итог:** для linguistic acceptability classification в текущей постановке предпочтительна instruction-tuned версия; base может использоваться при ограниченных ресурсах, но с ожидаемо более низким или менее стабильным качеством.